In [1]:
import requests
import pandas as pd

url = "http://127.0.0.1:8000/upload_config_from_csv/1/"

# input_csv = "phi_biogen_doc.csv"
# input_csv = "phi_lilly_doc.csv"
input_csv = "emd_serono_phi.csv"
# input_csv = "phi_referral.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

response = requests.post(url, files=files)
print(response.status_code, response.json())

200 {'success': True, 'message': 'Configuration uploaded successfully.'}


In [2]:
df = pd.read_csv(input_csv)
tables = list(df['TABLE_NAME'].unique())
len(tables)

52

In [3]:
# df = df[df['TABLE_NAME'].isin(['surgicalhistory', 'structsocialhistory', 'vitalshistory'])]
# df.shape, df['TABLE_NAME'].nunique()
# df.to_csv("phi_emd_temp.csv", index=False)

In [4]:
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

37

In [5]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

15

In [6]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

3

In [ ]:
import os
import django
import sys

# ---------------------------------------------------------------------------
# Adjust this ONE line to point to the deIdentification/ package root.
# ---------------------------------------------------------------------------
DEID_ROOT = os.path.abspath(
    os.path.join(os.path.dirname(os.path.abspath("__file__")), "../../deIdentification")
)
if DEID_ROOT not in sys.path:
    sys.path.insert(0, DEID_ROOT)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import (
    create_deidentification_task,
    create_deidentification_tasks_parallel,   # NEW: for large tables
)
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process_df.main import start_de_identification_for_table
from nd_api.views.db_views import run_stats_generation_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [9]:
to = TableDetailsModel.objects.get(table_name="edi_inv_cpt")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_cpt',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [10]:
to = TableDetailsModel.objects.get(table_name="edi_inv_diagnosis")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_diagnosis',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [11]:
# to = TableDetailsModel.objects.get(table_name="hcfa")
# to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
#    'source_column': 'InvId',
#    'reference_table': 'edi_invoice'}],
#  'source_table': 'hcfa',
#  'destination_column': 'PatientId',
#  'destination_column_type': 'patient_id'}
# to.save()

In [12]:
to = TableDetailsModel.objects.get(table_name="hl7labdatadetail")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'ReportId',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labdatadetail',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [13]:
to = TableDetailsModel.objects.get(table_name="hl7labnotes")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'reportid',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labnotes',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [14]:
to = TableDetailsModel.objects.get(table_name="labdataex")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'reportId',
   'reference_table': 'labdata'}],
 'source_table': 'labdataex',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [3]:
TableDetailsModel.objects.filter(table_status=0).count()

0

In [4]:
TableDetailsModel.objects.filter(table_status=1).count()

52

In [5]:
TableDetailsModel.objects.filter(table_status=2).count()

8

In [6]:
TableDetailsModel.objects.filter(table_status=3).count()

0

In [19]:
Task.objects.filter(status=0).count()

72574

In [ ]:
Task.objects.filter(status=1).count()

0

: 

In [16]:
Task.objects.filter(status=2).count()

8

In [10]:
Task.objects.filter(status=-1).count()

0

In [23]:
# Task.objects.filter(status=0).update(failure_count=0)
# Task.objects.filter(status=0).update(status=-1)

In [24]:
# Chain.objects.all().delete()

In [25]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION, settings.DEFAULT_OFFSET_VALUE

(10000, 34)

In [26]:
# tables = [table_obj.table_name for table_obj in TableDetailsModel.objects.filter(table_status=0)]
# len(tables)

In [27]:
# tables.remove("progressnotes_decryptfinal")
# tables = ['referraldetail']
completed = []
failed_tables = []
len(tables), len(completed), len(failed_tables)

(52, 0, 0)

In [ ]:
# -----------------------------------------------------------------------
# Tune these two values to match your worker / table setup.
#
# LARGE_TABLE_THRESHOLD  – tables whose ID range (MAX id − MIN id + 1) is
#   larger than this value are split into PARALLEL_TASKS_COUNT disjoint
#   keyset-ranged tasks so multiple workers can run simultaneously.
#   No stats generation is required — only two cheap index lookups
#   (MIN / MAX) are made at task-creation time.
#
# PARALLEL_TASKS_COUNT   – number of parallel tasks (and therefore workers)
#   that will process a large table concurrently.
# -----------------------------------------------------------------------
LARGE_TABLE_THRESHOLD = 5_000_000   # ID-range size above which → parallel
PARALLEL_TASKS_COUNT  = 4           # parallel sub-tasks for large tables

totalrun = 0
for table in tables:
    print(table)
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()

    # create_deidentification_tasks_parallel handles the single-vs-parallel
    # decision internally using a MIN/MAX query.  No rows_count needed.
    tasks, chain = create_deidentification_tasks_parallel(
        table_obj=tableobj,
        delete_table=True,
        num_parallel=PARALLEL_TASKS_COUNT,
        large_table_threshold=LARGE_TABLE_THRESHOLD,
    )

    totalrun += 1
    print(f"  tasks_created={len(tasks)}")

2026-02-21 10:06:16,751 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-21 10:06:16,757 - deIdentification.nd_logger - WARNING - Table cptcode_base does not exist and cannot be dropped.
2026-02-21 10:06:16,757 - deIdentification.nd_logger - INFO - Dropping ignore rows for cptcode_base, deid_prod


cptcode_base
[]
[]


2026-02-21 10:06:17,118 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-21 10:06:17,123 - deIdentification.nd_logger - WARNING - Table oldrxmain_addlinfo does not exist and cannot be dropped.
2026-02-21 10:06:17,124 - deIdentification.nd_logger - INFO - Dropping ignore rows for oldrxmain_addlinfo, deid_prod


24276
oldrxmain_addlinfo
[]
['EncounterId']


2026-02-21 10:06:42,792 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-21 10:06:42,816 - deIdentification.nd_logger - WARNING - Table allergies does not exist and cannot be dropped.
2026-02-21 10:06:42,816 - deIdentification.nd_logger - INFO - Dropping ignore rows for allergies, deid_prod


...

[]
['EncounterId']


2026-02-21 12:14:25,892 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-21 12:14:25,897 - deIdentification.nd_logger - WARNING - Table structured_data does not exist and cannot be dropped.
2026-02-21 12:14:25,897 - deIdentification.nd_logger - INFO - Dropping ignore rows for structured_data, deid_prod


405
structured_data
[]
['encounterid']
6669
labdataex


2026-02-21 12:14:27,772 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-21 12:14:27,776 - deIdentification.nd_logger - WARNING - Table labdataex does not exist and cannot be dropped.
2026-02-21 12:14:27,777 - deIdentification.nd_logger - INFO - Dropping ignore rows for labdataex, deid_prod


[]
[]
6211997
